# Discount and Revenue Validation

**Owner:** Omar Leopoldo  
**Assigned reviewer:** Sweta  
**Run after:** `03_gold_eda.ipynb`

Recalculates every line-level financial formula and reconciles discount bands, orders, and the revenue KPI.

This notebook is an owner-specific PySpark contribution. The owner must run it personally, inspect the displayed result, understand every assertion, and commit it from their own GitHub account.


## 1. Load the validated project tables


In [ ]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "analytics"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

sales_line = spark.table(f"{CATALOG}.{SCHEMA}.sales_line_gold")
orders_gold = spark.table(f"{CATALOG}.{SCHEMA}.orders_gold")
discount_impact = spark.table(f"{CATALOG}.{SCHEMA}.discount_impact_gold")
kpi = spark.table(f"{CATALOG}.{SCHEMA}.kpi_gold")

print(f"sales_line_gold: {sales_line.count():,}")
print(f"orders_gold: {orders_gold.count():,}")
print(f"discount_impact_gold: {discount_impact.count():,}")


## 2. Run owner-specific reconciliation and integrity checks


In [ ]:
# Recalculate every line-level financial field from its source columns.
financial_check = (
    sales_line
    .withColumn("ExpectedGross", F.round(F.col("Quantity") * F.col("UnitPrice"), 2))
    .withColumn("ExpectedDiscount", F.round(F.col("GrossRevenue") * F.col("DiscountPct"), 2))
    .withColumn("ExpectedNet", F.round(F.col("GrossRevenue") - F.col("DiscountAmount"), 2))
    .withColumn("ExpectedCost", F.round(F.col("Quantity") * F.col("UnitCost"), 2))
    .withColumn("ExpectedProfit", F.round(F.col("NetRevenue") - F.col("EstimatedCost"), 2))
)

formula_errors = financial_check.filter(
    (F.abs(F.col("GrossRevenue") - F.col("ExpectedGross")) > 0.01)
    | (F.abs(F.col("DiscountAmount") - F.col("ExpectedDiscount")) > 0.01)
    | (F.abs(F.col("NetRevenue") - F.col("ExpectedNet")) > 0.01)
    | (F.abs(F.col("EstimatedCost") - F.col("ExpectedCost")) > 0.01)
    | (F.abs(F.col("GrossProfit") - F.col("ExpectedProfit")) > 0.01)
)
assert formula_errors.count() == 0
assert sales_line.filter(F.col("NetRevenue") < 0).count() == 0

# Discount bands must cover every completed sales line exactly once.
assert discount_impact.agg(F.sum("LineItems")).first()[0] == sales_line.count()
sort_orders = [row["SortOrder"] for row in discount_impact.orderBy("SortOrder").select("SortOrder").collect()]
assert sort_orders == list(range(len(sort_orders)))

# Sales-line revenue, order revenue, and the executive KPI must reconcile.
line_revenue = sales_line.agg(F.round(F.sum("NetRevenue"), 2)).first()[0]
order_revenue = orders_gold.agg(F.round(F.sum("NetRevenue"), 2)).first()[0]
published_revenue = kpi.first()["TotalNetRevenue"]
assert abs(line_revenue - order_revenue) < 0.01
assert abs(line_revenue - published_revenue) < 0.01


## 3. Display the observed business result and success marker


In [ ]:
print(f"Validated sales lines: {sales_line.count():,}")
print(f"Reconciled net revenue: ${line_revenue:,.2f}")
display(
    discount_impact.orderBy("SortOrder").select(
        "DiscountBand", "LineItems", "AverageQuantity", "NetRevenue", "SortOrder"
    )
)

print("OMAR_DISCOUNT_REVENUE_VALIDATION_PASSED")


## What the owner must be able to explain

- Which tables were compared and why.
- What each assertion protects against.
- What the displayed result means for FreshRoute.
- Why the final success marker `OMAR_DISCOUNT_REVENUE_VALIDATION_PASSED` only prints after every check passes.
